# 7: Ablation Study
## RAG-Based Medical QA System — PubMedQA
---
**What this notebook does:**
- Compares Fixed vs Recursive chunking strategies
- Compares Semantic-Only vs Hybrid+Reranking retrieval
- Runs Faithfulness + Relevancy evaluation for each variation
- Produces the final ablation comparison table


## 7.1 Install Libraries

In [ ]:
!pip install groq sentence-transformers pinecone rank-bm25 pandas tqdm -q
print('Done!')

Done!


## 7.2 Imports & API Keys

In [ ]:
import json
import time
import re
import pickle
import numpy as np
import pandas as pd
from collections import defaultdict
from groq import Groq
from sentence_transformers import SentenceTransformer, CrossEncoder
from sklearn.metrics.pairwise import cosine_similarity
from pinecone import Pinecone

# ============================================================
#  FILL IN YOUR KEYS
# ============================================================
PINECONE_API_KEY = 'pcsk_4EURsR_7vhX2ipcBnChK6GbiyWKXFFWTgXjCTezhT1MuZufSHd5ehrHNdJQsRjHuBrvuha'
GROQ_API_KEY     = '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'
INDEX_NAME       = 'pubmed-rag'
GROQ_MODEL       = 'llama-3.3-70b-versatile'
# ============================================================

groq_client = Groq(api_key=GROQ_API_KEY)
print('Imports done!')

Imports done!


## 7.3 Upload Files

In [ ]:
from google.colab import files
print('Upload ALL of these files:')
print('  1. bm25_index.pkl')
print('  2. chunks_fixed.json')
print('  3. evaluation_results.json  (from Step 6)')
uploaded = files.upload()

Upload ALL of these files:
  1. bm25_index.pkl
  2. chunks_fixed.json
  3. evaluation_results.json  (from Step 6)


Saving evaluation_results.json to evaluation_results (1).json
Saving bm25_index.pkl to bm25_index (1).pkl
Saving chunks_fixed.json to chunks_fixed (1).json


## 7.4 Load Everything

In [ ]:
# Load BM25 (recursive chunks — our main strategy)
with open('bm25_index.pkl', 'rb') as f:
    bm25_data = pickle.load(f)
bm25_index       = bm25_data['bm25']
recursive_chunks = bm25_data['chunks']
print(f'Recursive chunks loaded: {len(recursive_chunks)} ✅')

# Load fixed chunks
with open('chunks_fixed.json', 'r') as f:
    fixed_chunks = json.load(f)
print(f'Fixed chunks loaded: {len(fixed_chunks)} ✅')

# Load Step 6 evaluation results (recursive + hybrid baseline)
with open('evaluation_results.json', 'r') as f:
    baseline_results = json.load(f)
print(f'Baseline evaluation loaded: {len(baseline_results)} queries ✅')

# Load models
print('Loading models...')
bi_encoder    = SentenceTransformer('all-MiniLM-L6-v2')
cross_encoder = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')
embedder      = bi_encoder  # same model used for relevancy scoring
print('Models loaded ✅')

# Connect Pinecone
pc    = Pinecone(api_key=PINECONE_API_KEY)
index = pc.Index(INDEX_NAME)
print(f'Pinecone connected ✅')

Recursive chunks loaded: 11605 ✅
Fixed chunks loaded: 8771 ✅
Baseline evaluation loaded: 5 queries ✅
Loading models...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Models loaded ✅
Pinecone connected ✅


## 7.5 Define All Helper Functions

In [ ]:
# ── Retrieval functions ──────────────────────────────────────────────────────

def semantic_search(query, bi_encoder, pinecone_index, top_k=20):
    query_emb = bi_encoder.encode([query], normalize_embeddings=True,
                                   convert_to_numpy=True)[0].tolist()
    results   = pinecone_index.query(vector=query_emb, top_k=top_k,
                                      include_metadata=True)
    return [{
        'chunk_id': m['id'], 'text': m['metadata']['text'],
        'score': m['score'], 'doc_id': m['metadata'].get('doc_id',''),
        'pubid': m['metadata'].get('pubid','')
    } for m in results['matches']]


def bm25_search(query, bm25_index, chunks, top_k=20):
    scores      = bm25_index.get_scores(query.lower().split())
    top_indices = np.argsort(scores)[::-1][:top_k]
    return [{
        'chunk_id': chunks[i]['chunk_id'], 'text': chunks[i]['text'],
        'score': float(scores[i]), 'doc_id': chunks[i]['doc_id'],
        'pubid': chunks[i]['pubid']
    } for i in top_indices if scores[i] > 0]


def rrf_fusion(sem_hits, bm25_hits, k=60, top_k=20):
    rrf_scores = defaultdict(float)
    chunk_data = {}
    for rank, hit in enumerate(sem_hits):
        rrf_scores[hit['chunk_id']] += 1.0 / (k + rank + 1)
        chunk_data[hit['chunk_id']]  = hit
    for rank, hit in enumerate(bm25_hits):
        rrf_scores[hit['chunk_id']] += 1.0 / (k + rank + 1)
        if hit['chunk_id'] not in chunk_data:
            chunk_data[hit['chunk_id']] = hit
    sorted_ids = sorted(rrf_scores, key=lambda x: rrf_scores[x], reverse=True)
    results = []
    for cid in sorted_ids[:top_k]:
        r = chunk_data[cid].copy()
        r['rrf_score'] = rrf_scores[cid]
        results.append(r)
    return results


def rerank(query, hits, cross_encoder, top_k=5):
    if not hits: return []
    scores = cross_encoder.predict([[query, h['text']] for h in hits])
    for h, s in zip(hits, scores): h['cross_score'] = float(s)
    return sorted(hits, key=lambda x: x['cross_score'], reverse=True)[:top_k]


# ── Generation & Evaluation functions ───────────────────────────────────────

def build_rag_prompt(query, chunks):
    ctx = ''.join([f'[{i+1}] (PubMed ID: {c["pubid"]})\n{c["text"]}\n\n'
                   for i, c in enumerate(chunks)])
    sys = ('You are a medical research assistant. Answer using ONLY the provided '
           'PubMed context. Cite [1],[2] etc. If insufficient info, say so.')
    usr = f'CONTEXT:\n{ctx}QUESTION: {query}\n\nANSWER:'
    return sys, usr


def generate_answer(query, chunks, groq_client):
    sys_msg, usr_msg = build_rag_prompt(query, chunks)
    t0 = time.time()
    resp = groq_client.chat.completions.create(
        model=GROQ_MODEL,
        messages=[{'role':'system','content':sys_msg},
                  {'role':'user','content':usr_msg}],
        max_tokens=400, temperature=0.1
    )
    return resp.choices[0].message.content.strip(), round((time.time()-t0)*1000,1)


def extract_claims(answer, groq_client):
    prompt = (f'Extract all distinct atomic factual claims from this answer as a '
              f'numbered list. Each claim = one verifiable fact.\n\nANSWER:\n{answer}\n\nCLAIMS:')
    resp = groq_client.chat.completions.create(
        model=GROQ_MODEL,
        messages=[{'role':'user','content':prompt}],
        max_tokens=300, temperature=0.0
    )
    claims = []
    for line in resp.choices[0].message.content.strip().split('\n'):
        c = re.sub(r'^\d+[\.\)\:]\s*','',line.strip()).strip()
        if c and len(c) > 10: claims.append(c)
    return claims


def verify_claim(claim, context, groq_client):
    prompt = (f'CONTEXT:\n{context}\n\nCLAIM:\n{claim}\n\n'
              f'Is this claim directly supported by the context?\n'
              f'VERDICT: [SUPPORTED or NOT SUPPORTED]\nREASON: [one sentence]')
    resp = groq_client.chat.completions.create(
        model=GROQ_MODEL,
        messages=[{'role':'user','content':prompt}],
        max_tokens=80, temperature=0.0
    )
    raw     = resp.choices[0].message.content.strip()
    verdict = 'NOT SUPPORTED'
    for line in raw.split('\n'):
        if 'VERDICT' in line.upper():
            verdict = 'SUPPORTED' if 'NOT' not in line.upper() and 'SUPPORTED' in line.upper() else 'NOT SUPPORTED'
    return verdict


def faithfulness_score(answer, chunks, groq_client):
    context = '\n\n'.join([f'[{i+1}] {c["text"]}' for i,c in enumerate(chunks)])
    claims  = extract_claims(answer, groq_client)
    if not claims: return 0.0, 0, 0
    supported = 0
    for claim in claims:
        v = verify_claim(claim, context, groq_client)
        if v == 'SUPPORTED': supported += 1
        time.sleep(0.3)
    return round(supported/len(claims)*100, 1), supported, len(claims)


def relevancy_score(query, answer, embedder, groq_client):
    prompt = (f'Generate exactly 3 questions that this answer is responding to.\n'
              f'Return a numbered list only.\n\nANSWER:\n{answer}\n\nQUESTIONS:')
    resp = groq_client.chat.completions.create(
        model=GROQ_MODEL,
        messages=[{'role':'user','content':prompt}],
        max_tokens=150, temperature=0.3
    )
    questions = []
    for line in resp.choices[0].message.content.strip().split('\n'):
        c = re.sub(r'^\d+[\.\)\:]\s*','',line.strip()).strip()
        if c and len(c) > 10: questions.append(c)
    questions = questions[:3]
    if not questions: return 0.0
    q_emb  = embedder.encode([query], normalize_embeddings=True)
    g_embs = embedder.encode(questions, normalize_embeddings=True)
    sims   = cosine_similarity(q_emb, g_embs)[0]
    return round(float(np.mean(sims))*100, 1)


print('All helper functions defined ✅')

All helper functions defined ✅


## 7.6 Define Test Queries
Same queries used in Steps 5 & 6 for fair comparison.

In [ ]:
TEST_QUERIES = [
    'Does aspirin reduce cardiovascular risk?',
    'What is the effect of metformin on type 2 diabetes?',
    'Does exercise reduce breast cancer risk?',
    'Are statins effective for lowering LDL cholesterol?',
    'Is cognitive behavioral therapy effective for depression?'
]

# Baseline scores from Step 6 (Recursive + Hybrid) — already computed!
baseline_faithfulness = [r['faithfulness']['faithfulness_score'] for r in baseline_results]
baseline_relevancy    = [r['relevancy']['relevancy_score']       for r in baseline_results]

print('Test queries set ✅')
print(f'Baseline (Recursive+Hybrid) Faithfulness: {baseline_faithfulness}')
print(f'Baseline (Recursive+Hybrid) Relevancy   : {baseline_relevancy}')

Test queries set ✅
Baseline (Recursive+Hybrid) Faithfulness: [85.7, 100.0, 80.0, 100.0, 100.0]
Baseline (Recursive+Hybrid) Relevancy   : [82.9, 85.6, 87.2, 87.6, 83.6]


## 7.7 Ablation Variation 1: Fixed Chunking + Hybrid Retrieval
Same retrieval strategy, different chunking. Uses BM25 built on fixed chunks.

In [ ]:
from rank_bm25 import BM25Okapi

# Build BM25 on fixed chunks
print('Building BM25 index on fixed chunks...')
fixed_tokenized = [c['text'].lower().split() for c in fixed_chunks]
fixed_bm25      = BM25Okapi(fixed_tokenized)
print(f'Fixed BM25 ready over {len(fixed_chunks)} chunks ✅')

fixed_hybrid_faith = []
fixed_hybrid_rel   = []

for i, query in enumerate(TEST_QUERIES):
    print(f'\n[{i+1}/5] {query}')

    # Retrieve using fixed chunks
    sem_hits  = semantic_search(query, bi_encoder, index, top_k=20)
    bm25_hits = bm25_search(query, fixed_bm25, fixed_chunks, top_k=20)
    fused     = rrf_fusion(sem_hits, bm25_hits, top_k=10)
    final     = rerank(query, fused, cross_encoder, top_k=5)

    # Generate answer
    answer, _ = generate_answer(query, final, groq_client)
    print(f'  Answer: {answer[:120]}...')

    # Evaluate
    f_score, _, _ = faithfulness_score(answer, final, groq_client)
    r_score       = relevancy_score(query, answer, embedder, groq_client)

    print(f'  Faithfulness: {f_score}% | Relevancy: {r_score}%')
    fixed_hybrid_faith.append(f_score)
    fixed_hybrid_rel.append(r_score)
    time.sleep(1.0)

print('\nVariation 1 complete ✅')

Building BM25 index on fixed chunks...
Fixed BM25 ready over 8771 chunks ✅

[1/5] Does aspirin reduce cardiovascular risk?
  Answer: According to [2], clinical trial evidence has supported aspirin use in the secondary prevention of cardiovascular diseas...
  Faithfulness: 100.0% | Relevancy: 87.1%

[2/5] What is the effect of metformin on type 2 diabetes?
  Answer: There is insufficient information in the provided context to determine the effect of metformin on type 2 diabetes. The c...
  Faithfulness: 85.7% | Relevancy: 89.6%

[3/5] Does exercise reduce breast cancer risk?
  Answer: There is insufficient information in the provided context to directly answer whether exercise reduces breast cancer risk...
  Faithfulness: 71.4% | Relevancy: 93.4%

[4/5] Are statins effective for lowering LDL cholesterol?
  Answer: Yes, statins are effective for lowering LDL cholesterol. According to [1] and [2], non-adjusted LDL levels decreased wit...
  Faithfulness: 87.5% | Relevancy: 85.6%

[5/5] Is 

## 7.8 Ablation Variation 2: Recursive Chunking + Semantic-Only (No BM25, No Reranking)
Same chunking as baseline, but retrieval uses only semantic search — no hybrid, no reranking.

In [ ]:
semantic_only_faith = []
semantic_only_rel   = []

for i, query in enumerate(TEST_QUERIES):
    print(f'\n[{i+1}/5] {query}')

    # Semantic-only retrieval (no BM25, no RRF, no CrossEncoder)
    final = semantic_search(query, bi_encoder, index, top_k=5)

    # Generate answer
    answer, _ = generate_answer(query, final, groq_client)
    print(f'  Answer: {answer[:120]}...')

    # Evaluate
    f_score, _, _ = faithfulness_score(answer, final, groq_client)
    r_score       = relevancy_score(query, answer, embedder, groq_client)

    print(f'  Faithfulness: {f_score}% | Relevancy: {r_score}%')
    semantic_only_faith.append(f_score)
    semantic_only_rel.append(r_score)
    time.sleep(1.0)

print('\nVariation 2 complete ✅')


[1/5] Does aspirin reduce cardiovascular risk?
  Answer: According to [1], clinical trial evidence has supported aspirin use in the secondary prevention of cardiovascular diseas...
  Faithfulness: 60.0% | Relevancy: 88.6%

[2/5] What is the effect of metformin on type 2 diabetes?
  Answer: According to [1], metformin therapy up to 21 g led to normalization of blood glucose levels and liver enzymes, and a wei...
  Faithfulness: 87.5% | Relevancy: 82.0%

[3/5] Does exercise reduce breast cancer risk?
  Answer: There is insufficient information in the provided context to directly answer whether exercise reduces breast cancer risk...
  Faithfulness: 50.0% | Relevancy: 94.2%

[4/5] Are statins effective for lowering LDL cholesterol?
  Answer: Yes, statins are effective for lowering LDL cholesterol. According to [2], atorvastatin reduced LDL levels, and [3] show...
  Faithfulness: 75.0% | Relevancy: 84.2%

[5/5] Is cognitive behavioral therapy effective for depression?
  Answer: Based on th

## 7.9 Build Final Ablation Comparison Table

In [ ]:
ablation_data = [
    {
        'Configuration'   : 'Semantic-Only (Recursive)',
        'Chunking'        : 'Recursive',
        'Retrieval'       : 'Semantic Only',
        'Re-ranking'      : 'No',
        'Avg Faith (%)'   : round(np.mean(semantic_only_faith), 1),
        'Avg Relevancy (%)': round(np.mean(semantic_only_rel), 1),
    },
    {
        'Configuration'   : 'Hybrid+Rerank (Fixed)',
        'Chunking'        : 'Fixed (200w)',
        'Retrieval'       : 'Hybrid (BM25+Semantic+RRF)',
        'Re-ranking'      : 'CrossEncoder',
        'Avg Faith (%)'   : round(np.mean(fixed_hybrid_faith), 1),
        'Avg Relevancy (%)': round(np.mean(fixed_hybrid_rel), 1),
    },
    {
        'Configuration'   : 'Hybrid+Rerank (Recursive) ← BEST',
        'Chunking'        : 'Recursive (800c)',
        'Retrieval'       : 'Hybrid (BM25+Semantic+RRF)',
        'Re-ranking'      : 'CrossEncoder',
        'Avg Faith (%)'   : round(np.mean(baseline_faithfulness), 1),
        'Avg Relevancy (%)': round(np.mean(baseline_relevancy), 1),
    },
]

df_ablation = pd.DataFrame(ablation_data)

print('=== ABLATION STUDY TABLE (for your report Section 2B) ===')
print(df_ablation.to_string(index=False))
print('\n(This table goes into your Ablation Study section)')

=== ABLATION STUDY TABLE (for your report Section 2B) ===
                   Configuration         Chunking                  Retrieval   Re-ranking  Avg Faith (%)  Avg Relevancy (%)
       Semantic-Only (Recursive)        Recursive              Semantic Only           No           72.3               87.2
           Hybrid+Rerank (Fixed)     Fixed (200w) Hybrid (BM25+Semantic+RRF) CrossEncoder           88.9               87.7
Hybrid+Rerank (Recursive) ← BEST Recursive (800c) Hybrid (BM25+Semantic+RRF) CrossEncoder           93.1               85.4

(This table goes into your Ablation Study section)


## 7.10 Per-Query Breakdown

In [ ]:
print('=== PER-QUERY ABLATION BREAKDOWN ===')
print(f'{"Query":<45} {"SemanticF":>9} {"SemanticR":>9} {"FixedF":>7} {"FixedR":>7} {"RecursF":>7} {"RecursR":>7}')
print('-' * 95)

for i, q in enumerate(TEST_QUERIES):
    print(
        f'{q[:44]:<45} '
        f'{semantic_only_faith[i]:>8.1f}% '
        f'{semantic_only_rel[i]:>8.1f}% '
        f'{fixed_hybrid_faith[i]:>6.1f}% '
        f'{fixed_hybrid_rel[i]:>6.1f}% '
        f'{baseline_faithfulness[i]:>6.1f}% '
        f'{baseline_relevancy[i]:>6.1f}%'
    )

print('-' * 95)
print(
    f'{"AVERAGE":<45} '
    f'{np.mean(semantic_only_faith):>8.1f}% '
    f'{np.mean(semantic_only_rel):>8.1f}% '
    f'{np.mean(fixed_hybrid_faith):>6.1f}% '
    f'{np.mean(fixed_hybrid_rel):>6.1f}% '
    f'{np.mean(baseline_faithfulness):>6.1f}% '
    f'{np.mean(baseline_relevancy):>6.1f}%'
)

=== PER-QUERY ABLATION BREAKDOWN ===
Query                                         SemanticF SemanticR  FixedF  FixedR RecursF RecursR
-----------------------------------------------------------------------------------------------
Does aspirin reduce cardiovascular risk?          60.0%     88.6%  100.0%   87.1%   85.7%   82.9%
What is the effect of metformin on type 2 di      87.5%     82.0%   85.7%   89.6%  100.0%   85.6%
Does exercise reduce breast cancer risk?          50.0%     94.2%   71.4%   93.4%   80.0%   87.2%
Are statins effective for lowering LDL chole      75.0%     84.2%   87.5%   85.6%  100.0%   87.6%
Is cognitive behavioral therapy effective fo      88.9%     86.9%  100.0%   82.9%  100.0%   83.6%
-----------------------------------------------------------------------------------------------
AVERAGE                                           72.3%     87.2%   88.9%   87.7%   93.1%   85.4%


## 7.11 Save Results

In [ ]:
ablation_results = {
    'semantic_only' : {'faithfulness': semantic_only_faith, 'relevancy': semantic_only_rel},
    'fixed_hybrid'  : {'faithfulness': fixed_hybrid_faith,  'relevancy': fixed_hybrid_rel},
    'recursive_hybrid': {'faithfulness': baseline_faithfulness, 'relevancy': baseline_relevancy}
}

with open('ablation_results.json', 'w') as f:
    json.dump(ablation_results, f, indent=2)
print('ablation_results.json saved ✅')

df_ablation.to_csv('ablation_table.csv', index=False)
print('ablation_table.csv saved ✅')

print('\nStep 7 Complete! ✅')
print('Next: Step 8 — Build Streamlit App + Deploy to HuggingFace Spaces')

ablation_results.json saved ✅
ablation_table.csv saved ✅

Step 7 Complete! ✅
Next: Step 8 — Build Streamlit App + Deploy to HuggingFace Spaces


In [ ]:
from google.colab import files
files.download('ablation_results.json')
files.download('ablation_table.csv')
print('Downloaded!')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Downloaded!
